<a href="https://colab.research.google.com/github/zeyneptass/vector-databases-and-rag-fundamentals/blob/main/RecommenderSystems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pinecone sentence-transformers openai tqdm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 13.4 MB/s eta 0:00:00


In [ ]:
import warnings
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata

# DeepSeek API anahtarını Colab Secrets'tan al
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

# DeepSeek istemcisini OpenAI formatında oluştur
client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

print("DeepSeek istemcisi hazır.")

DeepSeek istemcisi hazır.


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model boyutu: {model.get_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model boyutu: 384


In [ ]:
from datasets import load_dataset

# AG News veri setini yükle
dataset = load_dataset("fancyzhx/ag_news", split="train[:2000]")

# Haber metinlerini ve başlıklarını al
articles = [item['text'] for item in dataset]

print(f"Toplam {len(articles)} haber yüklendi.")
print("Örnek:", articles[0][:200])

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Toplam 2000 haber yüklendi.
Örnek: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [ ]:
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "recommender-demo"

if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=model.get_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

import time
while not pc.describe_index(index_name).status.ready:
    time.sleep(2)

index = pc.Index(index_name)
print("Dizin hazır.")

Dizin hazır.


In [ ]:
batch_size = 100

for i in tqdm(range(0, len(articles), batch_size)):
    batch = articles[i:i + batch_size]
    embeddings = model.encode(batch).tolist()

    vectors_to_upsert = []
    for j, (text, embedding) in enumerate(zip(batch, embeddings)):
        vector_id = f"news_{i + j}"
        metadata = {"text": text[:1000]}  # İlk 1000 karakter
        vectors_to_upsert.append((vector_id, embedding, metadata))

    index.upsert(vectors=vectors_to_upsert)

print("Veriler yüklendi!")

  0%|          | 0/20 [00:00<?, ?it/s]

Veriler yüklendi!


In [ ]:
def get_recommendations(query_text, top_k=5):
    """
    Bir sorgu metnini alır, vektöre çevirir ve Pinecone'da en benzer makaleleri arar.
    """
    # Sorguyu vektöre çevir
    query_embedding = model.encode(query_text).tolist()

    # Pinecone'da ara
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    # Sonuçları yazdır
    print(f"\nSorgu: '{query_text}'")
    print("-" * 70)
    for match in results['matches']:
        score = match['score']
        text = match['metadata']['text'][:120]
        print(f"Benzerlik: {score:.4f} | {text}...")
    print("-" * 70)

    return results

In [ ]:
# Test 1: Teknoloji haberleri
get_recommendations("What is the latest technology news?")

# Test 2: Spor haberleri
get_recommendations("Who won the football match?")

# Test 3: İş dünyası haberleri
get_recommendations("Stock market prices today")


Sorgu: 'What is the latest technology news?'
----------------------------------------------------------------------
Benzerlik: 0.4871 | Group to Propose New High-Speed Wireless Format (Reuters) Reuters - A group of technology companies\including Texas Inst...
Benzerlik: 0.4853 | The  #39;i #39; in  #39;Internet #39; It #39;s not exactly an earth-shaking development but I expect that Wired News #39...
Benzerlik: 0.4559 | News: Technology Already Exists to Stabilize Global Warming Existing technologies could stop the escalation of global wa...
Benzerlik: 0.4337 | Tech market indeed soft, but HP woes one of a kind HP and Cisco spooked investors with recent comments about sales, but ...
Benzerlik: 0.4180 | The Internet Is Calling Web phone service is exploding. And maverick Jeffrey Citron lit the industry's fuse...
----------------------------------------------------------------------

Sorgu: 'Who won the football match?'
-------------------------------------------------------------------

QueryResponse(matches=[ScoredVector(id='news_5', score=0.590506613, values=[], metadata={'text': 'Stocks End Up, But Near Year Lows (Reuters) Reuters - Stocks ended slightly higher on Friday\\but stayed near lows for the year as oil prices surged past  #36;46\\a barrel, offsetting a positive outlook from computer maker\\Dell Inc. (DELL.O)'}), ScoredVector(id='news_1797', score=0.587839782, values=[], metadata={'text': 'Stocks Higher on Economic Data, Earnings  NEW YORK (Reuters) - U.S. stocks traded higher on Tuesday,  despite oil prices hitting a new high before falling back, as  U.S. economic reports showed an easing of inflationary pressure  and a sharp rebound in the housing market.'}), ScoredVector(id='news_915', score=0.565029204, values=[], metadata={'text': "Stocks Sharply Higher on Dip in Oil Prices NEW YORK - A drop in oil prices and upbeat outlooks from Wal-Mart and Lowe's prompted new bargain-hunting on Wall Street Monday, sending stocks sharply higher. The Dow climbed more

In [ ]:
def summarize_recommendations(query_text, top_k=3):
    """
    Bir sorguya en benzer makaleleri bulur ve DeepSeek ile özetler.
    """
    # Önerileri al
    results = index.query(
        vector=model.encode(query_text).tolist(),
        top_k=top_k,
        include_metadata=True
    )

    # Bulunan makaleleri birleştir
    context = "\n\n".join([match['metadata']['text'] for match in results['matches']])

    # DeepSeek için prompt hazırla
    prompt = f"""Aşağıdaki bağlama göre "{query_text}" sorusunu yanıtla.

Bağlam:
{context}

Soru: {query_text}

Yanıt:"""

    # DeepSeek'e gönder
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {"role": "system", "content": "Sen yardımcı bir asistansın. Verilen bağlama göre soruları yanıtla."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.7
    )

    return response.choices[0].message.content

# Test et
answer = summarize_recommendations("What is the latest technology news?")
print(answer)

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance (request_id: 8429e4bc-06ef-43ac-80bf-8abb604c18f8)', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}